This model will make use of the best overhead RGB only model which was the ViT making use of DINOv2. This will be combined with the side angle images in order to help the model get better at estimating the volume of the dish. This will also mean that the second branch will be based on the mass error term (new metadata set which contains this information). The val_loss will however still be based on the carbohydrate loss in order to align with the rest of the models created.

First make use of the CPU before for the model training to switch to the GPU

Settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import csv

Parameters

In [ ]:
DATASET_DIR = Path("/content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3")
METADATA_CSV = Path("/content/drive/MyDrive/Speciale/dataset/prepared_overhead_rgb_dataset/dish_carb_mass_split_metadata.csv")

OUTPUT_DIR = Path("/content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREP_DIR = OUTPUT_DIR / "prep_files"
PREP_DIR.mkdir(parents=True, exist_ok=True)

print("DATASET_DIR:", DATASET_DIR)
print("DATASET_DIR exists:", DATASET_DIR.exists())
print("METADATA_CSV exists:", METADATA_CSV.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)
print("PREP_DIR:", PREP_DIR)

DATASET_DIR: /content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3
DATASET_DIR exists: True
METADATA_CSV exists: True
OUTPUT_DIR: /content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux
PREP_DIR: /content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/prep_files


Loading metadata and dataset

In [ ]:
df_meta = pd.read_csv(METADATA_CSV)

print("Metadata shape:", df_meta.shape)
print(df_meta["split"].value_counts(dropna=False))
display(df_meta.head())

Metadata shape: (3262, 5)
split
train    2421
test      507
val       334
Name: count, dtype: int64


,dish_id,total_mass,total_carb,split,source_cafe
0,dish_1556575327,152.0,10.618,test,cafe1
1,dish_1557861216,1.0,0.000,test,cafe1
2,dish_1557862345,63.0,0.000,test,cafe1
3,dish_1557862696,77.0,3.850,test,cafe1
4,dish_1557862738,118.0,0.000,test,cafe1


In [ ]:
def find_frame_in_camera_dir(camera_dir: Path, frame_prefix: str):
    if not camera_dir.exists() or not camera_dir.is_dir():
        return None

    preferred_names = [
        f"{frame_prefix}.jpg",
        f"{frame_prefix}.jpeg",
        f"{frame_prefix}.png",
        f"{frame_prefix}.JPG",
        f"{frame_prefix}.JPEG",
        f"{frame_prefix}.PNG",
    ]

    for fname in preferred_names:
        p = camera_dir / fname
        if p.exists():
            return str(p)

    matches = []
    for p in sorted(camera_dir.iterdir()):
        if p.is_file() and p.stem.lower().startswith(frame_prefix.lower()):
            matches.append(p)

    if len(matches) == 0:
        return None

    return str(matches[0])

In [ ]:
def build_multiview_paths_model7_style(dish_id):
    dish_id = str(dish_id)
    dish_dir = DATASET_DIR / dish_id
    side_root = dish_dir / "side_angles"

    record = {
        "dish_id": dish_id,
        "rgb_path": str(dish_dir / "overhead" / "rgb.png"),
    }

    camera_map = {
        "camera_A": "a",
        "camera_B": "b",
        "camera_C": "c",
        "camera_D": "d",
    }

    for cam_name, cam_short in camera_map.items():
        cam_dir = side_root / cam_name
        record[f"side_{cam_short}_start_path"] = find_frame_in_camera_dir(cam_dir, "frame_start")
        record[f"side_{cam_short}_mid_path"]   = find_frame_in_camera_dir(cam_dir, "frame_mid")

    return record

In [ ]:
sample_dish_id = str(df_meta.iloc[0]["dish_id"])
sample_paths = build_multiview_paths_model7_style(sample_dish_id)

print("Sample dish:", sample_dish_id)
for k, v in sample_paths.items():
    print(k, "->", v)

Sample dish: dish_1556575327
dish_id -> dish_1556575327
rgb_path -> /content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3/dish_1556575327/overhead/rgb.png
side_a_start_path -> /content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3/dish_1556575327/side_angles/camera_A/frame_start.jpg
side_a_mid_path -> /content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3/dish_1556575327/side_angles/camera_A/frame_mid.jpg
side_b_start_path -> /content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3/dish_1556575327/side_angles/camera_B/frame_start.jpg
side_b_mid_path -> /content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3/dish_1556575327/side_angles/camera_B/frame_mid.jpg
side_c_start_path -> /content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3/dish_1556575327/side_angles/camera_C/frame_start.jpg
side_c_mid_path -> /content/drive/MyDrive/Speciale/dataset/nutrition5k_compactV3/dish_1556575327/side_angles/camera_C/frame_mid.jpg
side_d_start_path -> /content/dri

In [ ]:
records = []

for dish_id in df_meta["dish_id"].astype(str).tolist():
    records.append(build_multiview_paths_model7_style(dish_id))

paths_df = pd.DataFrame(records)

df = df_meta.copy()
df["dish_id"] = df["dish_id"].astype(str)
paths_df["dish_id"] = paths_df["dish_id"].astype(str)

df = df.merge(paths_df, on="dish_id", how="left")

display(df.head())
print("Shape after merge:", df.shape)

,dish_id,total_mass,total_carb,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,side_c_start_path,side_c_mid_path,side_d_start_path,side_d_mid_path
0,dish_1556575327,152.0,10.618,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
1,dish_1557861216,1.0,0.000,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
2,dish_1557862345,63.0,0.000,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
3,dish_1557862696,77.0,3.850,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
4,dish_1557862738,118.0,0.000,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...


Shape after merge: (3262, 14)


In [ ]:
side_path_columns = [
    "side_a_start_path",
    "side_a_mid_path",
    "side_b_start_path",
    "side_b_mid_path",
    "side_c_start_path",
    "side_c_mid_path",
    "side_d_start_path",
    "side_d_mid_path",
]

In [ ]:
def safe_exists(path_str):
    if pd.isna(path_str):
        return False
    try:
        return Path(path_str).exists()
    except Exception:
        return False

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

df["rgb_exists"] = df["rgb_path"].progress_apply(safe_exists)

for col in side_path_columns:
    df[f"{col}_exists"] = df[col].progress_apply(safe_exists)

display(df.head())

  0%|          | 0/3262 [00:00<?, ?it/s]

  0%|          | 0/3262 [00:00<?, ?it/s]

  0%|          | 0/3262 [00:00<?, ?it/s]

  0%|          | 0/3262 [00:00<?, ?it/s]

  0%|          | 0/3262 [00:00<?, ?it/s]

  0%|          | 0/3262 [00:00<?, ?it/s]

  0%|          | 0/3262 [00:00<?, ?it/s]

  0%|          | 0/3262 [00:00<?, ?it/s]

  0%|          | 0/3262 [00:00<?, ?it/s]

,dish_id,total_mass,total_carb,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,...,side_d_mid_path,rgb_exists,side_a_start_path_exists,side_a_mid_path_exists,side_b_start_path_exists,side_b_mid_path_exists,side_c_start_path_exists,side_c_mid_path_exists,side_d_start_path_exists,side_d_mid_path_exists
0,dish_1556575327,152.0,10.618,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,...,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,True,True,True,True,True,True,True,True
1,dish_1557861216,1.0,0.000,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,...,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,True,True,True,True,True,True,True,True
2,dish_1557862345,63.0,0.000,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,...,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,True,True,True,True,True,True,True,True
3,dish_1557862696,77.0,3.850,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,...,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,True,True,True,True,True,True,True,True
4,dish_1557862738,118.0,0.000,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,...,/content/drive/MyDrive/Speciale/dataset/nutrit...,True,True,True,True,True,True,True,True,True


Summary of dishes with missing side angle images

In [ ]:
summary_rows = []

summary_rows.append({
    "column": "rgb_path",
    "exists_count": int(df["rgb_exists"].sum()),
    "missing_count": int((~df["rgb_exists"]).sum())
})

for col in side_path_columns:
    exists_col = f"{col}_exists"
    summary_rows.append({
        "column": col,
        "exists_count": int(df[exists_col].sum()),
        "missing_count": int((~df[exists_col]).sum())
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,column,exists_count,missing_count
0,rgb_path,3262,0
1,side_a_start_path,2686,576
2,side_a_mid_path,2686,576
3,side_b_start_path,3082,180
4,side_b_mid_path,3082,180
5,side_c_start_path,2940,322
6,side_c_mid_path,2940,322
7,side_d_start_path,3094,168
8,side_d_mid_path,3094,168


Filtering so the dataset only contains complete dishes (meaning overhead and all side angle images)

In [ ]:
valid_mask = df["rgb_exists"].copy()

for col in side_path_columns:
    valid_mask &= df[f"{col}_exists"]

df_filtered = df[valid_mask].copy().reset_index(drop=True)

print("Original rows:", len(df))
print("Filtered rows:", len(df_filtered))
print(df_filtered["split"].value_counts(dropna=False))

Original rows: 3262
Filtered rows: 2629
split
train    1904
test      401
val       324
Name: count, dtype: int64


In [ ]:
columns_to_keep = [
    "dish_id",
    "total_carb",
    "total_mass",
    "split",
    "source_cafe",
    "rgb_path",
    "side_a_start_path",
    "side_a_mid_path",
    "side_b_start_path",
    "side_b_mid_path",
    "side_c_start_path",
    "side_c_mid_path",
    "side_d_start_path",
    "side_d_mid_path",
]

df_filtered = df_filtered[columns_to_keep].copy()
display(df_filtered.head())

,dish_id,total_carb,total_mass,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,side_c_start_path,side_c_mid_path,side_d_start_path,side_d_mid_path
0,dish_1556575327,10.618,152.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
1,dish_1557861216,0.000,1.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
2,dish_1557862345,0.000,63.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
3,dish_1557862696,3.850,77.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
4,dish_1557862738,0.000,118.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...


Splitting dataset

In [ ]:
train_df = df_filtered[df_filtered["split"] == "train"].copy().reset_index(drop=True)
val_df   = df_filtered[df_filtered["split"] == "val"].copy().reset_index(drop=True)
test_df  = df_filtered[df_filtered["split"] == "test"].copy().reset_index(drop=True)

print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))

Train: 1904
Val:   324
Test:  401


In [ ]:
train_csv = PREP_DIR / "train_rgb_all_side_mass_aux_v1.csv"
val_csv   = PREP_DIR / "val_rgb_all_side_mass_aux_v1.csv"
test_csv  = PREP_DIR / "test_rgb_all_side_mass_aux_v1.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

print(train_csv)
print(val_csv)
print(test_csv)

/content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/prep_files/train_rgb_all_side_mass_aux_v1.csv
/content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/prep_files/val_rgb_all_side_mass_aux_v1.csv
/content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/prep_files/test_rgb_all_side_mass_aux_v1.csv


SWITCH TO GPU FOR MODEL TRAINING

In [ ]:
!pip install -q keras-hub huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Imports

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import keras_hub

Model configuration

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH  = 224
BATCH_SIZE = 16
AUTOTUNE   = tf.data.AUTOTUNE

STAGE1_EPOCHS = 100
STAGE2_EPOCHS = 200

STAGE1_LR = 2e-4
STAGE2_LR = 5e-6

RGB_BACKBONE_PRESET = "dinov2_base"
SIDE_BACKBONE_PRESET = "dinov3_vit_base_lvd1689m"

MASS_LOSS_WEIGHT = 0.05

OUTPUT_DIR = Path("/content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux")
PREP_DIR = OUTPUT_DIR / "prep_files"

train_csv = PREP_DIR / "train_rgb_all_side_mass_aux_v1.csv"
val_csv   = PREP_DIR / "val_rgb_all_side_mass_aux_v1.csv"
test_csv  = PREP_DIR / "test_rgb_all_side_mass_aux_v1.csv"

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("GPU available:", tf.config.list_physical_devices("GPU"))

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Loading metadata and dataset

In [ ]:
train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
test_df  = pd.read_csv(test_csv)

print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))

display(train_df.head())

Train: 1904
Val:   324
Test:  401


,dish_id,total_carb,total_mass,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,side_c_start_path,side_c_mid_path,side_d_start_path,side_d_mid_path
0,dish_1556573514,1.219,23.0,train,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
1,dish_1556575014,3.906,62.0,train,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
2,dish_1556575083,5.760,64.0,train,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
3,dish_1556575124,0.952,28.0,train,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...
4,dish_1556575273,10.618,152.0,train,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...


In [ ]:
side_path_columns = [
    "side_a_start_path",
    "side_a_mid_path",
    "side_b_start_path",
    "side_b_mid_path",
    "side_c_start_path",
    "side_c_mid_path",
    "side_d_start_path",
    "side_d_mid_path",
]

In [ ]:
def load_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])

    image = tf.image.resize_with_pad(
        image,
        target_height=IMG_HEIGHT,
        target_width=IMG_WIDTH,
        method=tf.image.ResizeMethod.BILINEAR,
        antialias=True
    )

    image = tf.cast(image, tf.float32)
    return image

In [ ]:
def load_multiview_inputs(
    rgb_path,
    side_a_start_path, side_a_mid_path,
    side_b_start_path, side_b_mid_path,
    side_c_start_path, side_c_mid_path,
    side_d_start_path, side_d_mid_path,
    carb_label,
    mass_label
):
    rgb = load_image(rgb_path)

    side_a_start = load_image(side_a_start_path)
    side_a_mid   = load_image(side_a_mid_path)
    side_b_start = load_image(side_b_start_path)
    side_b_mid   = load_image(side_b_mid_path)
    side_c_start = load_image(side_c_start_path)
    side_c_mid   = load_image(side_c_mid_path)
    side_d_start = load_image(side_d_start_path)
    side_d_mid   = load_image(side_d_mid_path)

    carb_label = tf.cast(carb_label, tf.float32)
    mass_label = tf.cast(mass_label, tf.float32)

    return {
        "rgb_input": rgb,
        "side_a_start_input": side_a_start,
        "side_a_mid_input": side_a_mid,
        "side_b_start_input": side_b_start,
        "side_b_mid_input": side_b_mid,
        "side_c_start_input": side_c_start,
        "side_c_mid_input": side_c_mid,
        "side_d_start_input": side_d_start,
        "side_d_mid_input": side_d_mid,
    }, {
        "carb_output": carb_label,
        "mass_output": mass_label,
    }

In [ ]:
def make_dataset(dataframe, batch_size=BATCH_SIZE, shuffle=False):
    tensors = (
        dataframe["rgb_path"].astype(str).values,
        dataframe["side_a_start_path"].astype(str).values,
        dataframe["side_a_mid_path"].astype(str).values,
        dataframe["side_b_start_path"].astype(str).values,
        dataframe["side_b_mid_path"].astype(str).values,
        dataframe["side_c_start_path"].astype(str).values,
        dataframe["side_c_mid_path"].astype(str).values,
        dataframe["side_d_start_path"].astype(str).values,
        dataframe["side_d_mid_path"].astype(str).values,
        dataframe["total_carb"].values.astype("float32"),
        dataframe["total_mass"].values.astype("float32"),
    )

    ds = tf.data.Dataset.from_tensor_slices(tensors)

    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_multiview_inputs, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds

In [ ]:
train_dataset      = make_dataset(train_df, batch_size=BATCH_SIZE, shuffle=True)
validation_dataset = make_dataset(val_df, batch_size=BATCH_SIZE, shuffle=False)
test_dataset       = make_dataset(test_df, batch_size=BATCH_SIZE, shuffle=False)

print("Datasets created.")

Datasets created.


Inspecting one batch

In [ ]:
for batch_inputs, batch_labels in train_dataset.take(1):
    print("Input keys:", batch_inputs.keys())
    print("Label keys:", batch_labels.keys())
    print("RGB batch shape:", batch_inputs["rgb_input"].shape)
    print("Side A start shape:", batch_inputs["side_a_start_input"].shape)
    print("Carb labels shape:", batch_labels["carb_output"].shape)
    print("Mass labels shape:", batch_labels["mass_output"].shape)

Input keys: dict_keys(['rgb_input', 'side_a_start_input', 'side_a_mid_input', 'side_b_start_input', 'side_b_mid_input', 'side_c_start_input', 'side_c_mid_input', 'side_d_start_input', 'side_d_mid_input'])
Label keys: dict_keys(['carb_output', 'mass_output'])
RGB batch shape: (16, 224, 224, 3)
Side A start shape: (16, 224, 224, 3)
Carb labels shape: (16,)
Mass labels shape: (16,)


Loading in backbone and image converter of pretrained ViT

In [ ]:
rgb_image_converter = keras_hub.layers.ImageConverter.from_preset(
    RGB_BACKBONE_PRESET,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
)

side_image_converter = keras_hub.layers.ImageConverter.from_preset(
    SIDE_BACKBONE_PRESET,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
)

rgb_backbone = keras_hub.models.Backbone.from_preset(
    RGB_BACKBONE_PRESET,
    image_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
)

side_backbone = keras_hub.models.Backbone.from_preset(
    SIDE_BACKBONE_PRESET,
    image_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
)

rgb_backbone.trainable = False
side_backbone.trainable = False

100%|██████████| 1.13k/1.13k [00:00<00:00, 2.64MB/s]


100%|██████████| 978/978 [00:00<00:00, 1.87MB/s]


100%|██████████| 327M/327M [00:26<00:00, 13.0MB/s]


In [ ]:
def extract_backbone_tensor(x):
    if isinstance(x, dict):
        preferred_keys = [
            "sequence_output",
            "pooled_output",
            "encoder_output",
            "token_embeddings",
            "hidden_states"
        ]
        for key in preferred_keys:
            if key in x:
                return x[key]
        return list(x.values())[0]
    return x

Model architecture

In [ ]:
rgb_input = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="rgb_input")

side_a_start_input = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="side_a_start_input")
side_a_mid_input   = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="side_a_mid_input")
side_b_start_input = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="side_b_start_input")
side_b_mid_input   = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="side_b_mid_input")
side_c_start_input = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="side_c_start_input")
side_c_mid_input   = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="side_c_mid_input")
side_d_start_input = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="side_d_start_input")
side_d_mid_input   = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="side_d_mid_input")

In [ ]:
rgb_norm_1d = layers.LayerNormalization(name="rgb_norm_1d")
rgb_norm_flatten = layers.LayerNormalization(name="rgb_norm_flatten")
side_norm_1d = layers.LayerNormalization(name="side_norm_1d")
side_norm_flatten = layers.LayerNormalization(name="side_norm_flatten")

def encode_rgb(x):
    x = rgb_image_converter(x)
    x = rgb_backbone(x)
    x = extract_backbone_tensor(x)

    if len(x.shape) == 3:
        x = rgb_norm_1d(x)
        x = layers.GlobalAveragePooling1D()(x)
    else:
        x = rgb_norm_flatten(x)
        x = layers.Flatten()(x)
    return x

def encode_side(x):
    x = side_image_converter(x)
    x = side_backbone(x)
    x = extract_backbone_tensor(x)

    if len(x.shape) == 3:
        x = side_norm_1d(x)
        x = layers.GlobalAveragePooling1D()(x)
    else:
        x = side_norm_flatten(x)
        x = layers.Flatten()(x)
    return x

In [ ]:
rgb_feat = encode_rgb(rgb_input)

side_feats = [
    encode_side(side_a_start_input),
    encode_side(side_a_mid_input),
    encode_side(side_b_start_input),
    encode_side(side_b_mid_input),
    encode_side(side_c_start_input),
    encode_side(side_c_mid_input),
    encode_side(side_d_start_input),
    encode_side(side_d_mid_input),
]

In [ ]:
side_stack = layers.Lambda(
    lambda tensors: tf.stack(tensors, axis=1),
    name="side_stack"
)(side_feats)

side_feat = layers.Lambda(
    lambda x: tf.reduce_mean(x, axis=1),
    name="side_mean_pool"
)(side_stack)

In [ ]:
mass_branch = layers.Dropout(0.30, name="mass_dropout_1")(side_feat)
mass_branch = layers.Dense(256, activation="relu", name="mass_dense_256")(mass_branch)
mass_branch = layers.Dropout(0.30, name="mass_dropout_2")(mass_branch)
mass_branch = layers.Dense(64, activation="relu", name="mass_dense_64")(mass_branch)

mass_output = layers.Dense(1, activation="linear", name="mass_output")(mass_branch)

fusion = layers.Concatenate(name="fusion_concat")([rgb_feat, side_feat])

x = layers.Dropout(0.30, name="fusion_dropout_1")(fusion)
x = layers.Dense(256, activation="relu", name="dense_256")(x)
x = layers.Dropout(0.30, name="fusion_dropout_2")(x)
x = layers.Dense(64, activation="relu", name="dense_64")(x)

carb_output = layers.Dense(1, activation="linear", name="carb_output")(x)

model = keras.Model(
    inputs={
        "rgb_input": rgb_input,
        "side_a_start_input": side_a_start_input,
        "side_a_mid_input": side_a_mid_input,
        "side_b_start_input": side_b_start_input,
        "side_b_mid_input": side_b_mid_input,
        "side_c_start_input": side_c_start_input,
        "side_c_mid_input": side_c_mid_input,
        "side_d_start_input": side_d_start_input,
        "side_d_mid_input": side_d_mid_input,
    },
    outputs={
        "carb_output": carb_output,
        "mass_output": mass_output,
    },
    name="model_9_v1_rgb_plus_all_side_mass_aux"
)

model.summary()

Model: "model_9_v1_rgb_plus_all_side_mass_aux"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ side_a_start_input  │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_a_mid_input    │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_b_start_input  │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_b_mid_input    │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_c_start_input  │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_c_mid_input    │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_d_start_input  │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_d_mid_input    │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dinov3_image_conve… │ (None, 224, 224,  │          0 │ side_a_start_inp… │
│ (DINOV3ImageConver… │ 3)                │            │ side_a_mid_input… │
│                     │                   │            │ side_b_start_inp… │
│                     │                   │            │ side_b_mid_input… │
│                     │                   │            │ side_c_start_inp… │
│                     │                   │            │ side_c_mid_input… │
│                     │                   │            │ side_d_start_inp… │
│                     │                   │            │ side_d_mid_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rgb_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dinov3_backbone     │ (None, 201, 768)  │ 85,660,416 │ dinov3_image_con… │
│ (DINOV3Backbone)    │                   │            │ dinov3_image_con… │
│                     │                   │            │ dinov3_image_con… │
│                     │                   │            │ dinov3_image_con… │
│                     │                   │            │ dinov3_image_con… │
│                     │                   │            │ dinov3_image_con… │
│                     │                   │            │ dinov3_image_con… │
│                     │                   │            │ dinov3_image_con… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dinov2_image_conve… │ (None, 224, 224,  │          0 │ rgb_input[0][0]   │
│ (DINOV2ImageConver… │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_norm_1d        │ (None, 201, 768)  │      1,536 │ dinov3_backbone[

 Total params: 173,064,706 (660.19 MB)

 Trainable params: 626,434 (2.39 MB)

 Non-trainable params: 172,438,272 (657.80 MB)

Stage 1 training (only head of the model entire backbone frozen)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=STAGE1_LR),
    loss={
        "carb_output": "mse",
        "mass_output": "mse",
    },
    loss_weights={
        "carb_output": 1.0,
        "mass_output": MASS_LOSS_WEIGHT,
    },
    metrics={
        "carb_output": [
            keras.metrics.MeanAbsoluteError(name="mae"),
            keras.metrics.MeanSquaredError(name="mse"),
        ],
        "mass_output": [
            keras.metrics.MeanAbsoluteError(name="mae"),
            keras.metrics.MeanSquaredError(name="mse"),
        ],
    },
    jit_compile=False
)

print("Stage 1 model compiled.")

Stage 1 model compiled.


In [ ]:
stage1_best_path = OUTPUT_DIR / "best_model_stage1.weights.h5"

stage1_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(stage1_best_path),
        monitor="val_carb_output_loss",
        save_best_only=True,
        save_weights_only=True,
        mode="min",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_carb_output_loss",
        patience=50,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_carb_output_loss",
        factor=0.5,
        patience=35,
        min_lr=1e-6,
        mode="min",
        verbose=1
    )
]

print("Stage 1 callbacks ready.")
print("Best Stage 1 weights will be saved to:")
print(stage1_best_path)

Stage 1 callbacks ready.
Best Stage 1 weights will be saved to:
/content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/best_model_stage1.weights.h5


In [ ]:
history_stage1 = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=STAGE1_EPOCHS,
    callbacks=stage1_callbacks,
    verbose=1
)

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'pixel_values': 'pixel_values'}
Received: inputs=Tensor(shape=(None, 224, 224, 3))
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'images': 'images'}
Received: inputs=Tensor(shape=(None, 224, 224, 3))
  warnings.warn(msg)


119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step - carb_output_loss: 912.8517 - carb_output_mae: 14.1239 - carb_output_mse: 912.8517 - loss: 4705.6006 - mass_output_loss: 75854.9787 - mass_output_mae: 210.2432 - mass_output_mse: 75854.9787 
Epoch 1: val_carb_output_loss improved from None to 157.67302, saving model to /content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/best_model_stage1.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/best_model_stage1.weights.h5
119/119 ━━━━━━━━━━━━━━━━━━━━ 1717s 14s/step - carb_output_loss: 606.0806 - carb_output_mae: 12.2982 - carb_output_mse: 606.0806 - loss: 3513.7717 - mass_output_loss: 58153.8359 - mass_output_mae: 178.6692 - mass_output_mse: 58153.8359 - val_carb_output_loss: 157.6730 - val_carb_output_mae: 10.0237 - val_carb_output_mse: 159.4693 - val_loss: 1289.4602 - val_mass_output_loss: 22013.2227 - val_mass_output_mae: 117.6035 - val_m

Evaluating stage 1

In [ ]:
model.load_weights(str(stage1_best_path))
print("Loaded best Stage 1 weights.")

stage1_test_results = model.evaluate(test_dataset, verbose=1, return_dict=True)

print("\nStage 1 test results:")
print(stage1_test_results)

Loaded best Stage 1 weights.
26/26 ━━━━━━━━━━━━━━━━━━━━ 303s 11s/step - carb_output_loss: 95.6523 - carb_output_mae: 6.2110 - carb_output_mse: 80.9516 - loss: 199.5292 - mass_output_loss: 2318.1040 - mass_output_mae: 32.4311 - mass_output_mse: 2371.5510

Stage 1 test results:
{'carb_output_loss': 95.65231323242188, 'carb_output_mae': 6.210975170135498, 'carb_output_mse': 80.95161437988281, 'loss': 199.52915954589844, 'mass_output_loss': 2318.10400390625, 'mass_output_mae': 32.43112564086914, 'mass_output_mse': 2371.551025390625}


In [ ]:
stage1_predictions = model.predict(test_dataset, verbose=1)["carb_output"].reshape(-1)

stage1_test_df = test_df.copy().reset_index(drop=True)
stage1_test_df["prediction_stage1"] = stage1_predictions
stage1_test_df["abs_error_stage1"] = np.abs(
    stage1_test_df["total_carb"] - stage1_test_df["prediction_stage1"]
)

display(stage1_test_df.head())

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'pixel_values': 'pixel_values'}
Received: inputs=Tensor(shape=(16, 224, 224, 3))
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'images': 'images'}
Received: inputs=Tensor(shape=(16, 224, 224, 3))
  warnings.warn(msg)


26/26 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step


,dish_id,total_carb,total_mass,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,side_c_start_path,side_c_mid_path,side_d_start_path,side_d_mid_path,prediction_stage1,abs_error_stage1
0,dish_1556575327,10.618,152.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,16.537474,5.919474
1,dish_1557861216,0.000,1.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,4.456132,4.456132
2,dish_1557862345,0.000,63.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,3.904087,3.904087
3,dish_1557862696,3.850,77.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,4.856776,1.006776
4,dish_1557862738,0.000,118.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,8.042960,8.042960


Top 10 best and worst predictions for stage 1

In [ ]:
stage1_best_10 = stage1_test_df.sort_values("abs_error_stage1", ascending=True).head(10)
stage1_worst_10 = stage1_test_df.sort_values("abs_error_stage1", ascending=False).head(10)

stage1_best_10_csv = OUTPUT_DIR / "stage1_best_10.csv"
stage1_worst_10_csv = OUTPUT_DIR / "stage1_worst_10.csv"

stage1_best_10.to_csv(stage1_best_10_csv, index=False)
stage1_worst_10.to_csv(stage1_worst_10_csv, index=False)

display(stage1_best_10)
display(stage1_worst_10)

,dish_id,total_carb,total_mass,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,side_c_start_path,side_c_mid_path,side_d_start_path,side_d_mid_path,prediction_stage1,abs_error_stage1
49,dish_1558376801,0.795000,15.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,0.796074,0.001074
372,dish_1566849895,12.821001,153.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,12.809877,0.011124
163,dish_1561662814,1.529342,20.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,1.515391,0.013951
399,dish_1568401261,34.582966,443.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,34.566231,0.016735
184,dish_1562099053,12.395000,37.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,12.354160,0.040840
95,dish_1560368464,16.807095,275.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,16.897425,0.090330
398,dish_1568401233,32.332966,418.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,32.423515,0.090549
270,dish_1563909507,8.186116,151.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive

,dish_id,total_carb,total_mass,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,side_c_start_path,side_c_mid_path,side_d_start_path,side_d_mid_path,prediction_stage1,abs_error_stage1
120,dish_1560801020,61.690983,250.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,20.011377,41.679606
119,dish_1560800988,61.249092,230.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,23.395296,37.853796
121,dish_1560801041,63.245892,292.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,27.730942,35.514950
91,dish_1560367952,45.956913,221.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,11.422873,34.534040
92,dish_1560367980,47.243912,254.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,13.527180,33.716732
353,dish_1566501594,54.281998,109.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,21.808552,32.473446
259,dish_1563566909,46.750000,187.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,16.850647,29.899353
247,dish_1563478751,54.133282,545.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/conten

Stage 2 (unfreezing 30% of the backbone but lowering learning rate)

In [ ]:
model.load_weights(str(stage1_best_path))
print("Best Stage 1 weights loaded as the starting point for Stage 2.")

Best Stage 1 weights loaded as the starting point for Stage 2.


In [ ]:
rgb_backbone.trainable = True
side_backbone.trainable = True

rgb_num_layers = len(rgb_backbone.layers)
rgb_freeze_until = int(rgb_num_layers * 0.70)

for i, layer in enumerate(rgb_backbone.layers):
    layer.trainable = (i >= rgb_freeze_until)

side_num_layers = len(side_backbone.layers)
side_freeze_until = int(side_num_layers * 0.70)

for i, layer in enumerate(side_backbone.layers):
    layer.trainable = (i >= side_freeze_until)

print("Stage 2 unfreezing summary:")
print(f"RGB backbone total layers:   {rgb_num_layers}")
print(f"RGB frozen layers:          {rgb_freeze_until}")
print(f"RGB trainable layers:       {rgb_num_layers - rgb_freeze_until}")

print(f"Side backbone total layers: {side_num_layers}")
print(f"Side frozen layers:         {side_freeze_until}")
print(f"Side trainable layers:      {side_num_layers - side_freeze_until}")

Stage 2 unfreezing summary:
RGB backbone total layers:   4
RGB frozen layers:          2
RGB trainable layers:       2
Side backbone total layers: 5
Side frozen layers:         3
Side trainable layers:      2


In [ ]:
trainable_count = np.sum([np.prod(v.shape) for v in model.trainable_weights])
non_trainable_count = np.sum([np.prod(v.shape) for v in model.non_trainable_weights])

print("Trainable params:", int(trainable_count))
print("Non-trainable params:", int(non_trainable_count))

Trainable params: 170766082
Non-trainable params: 2298624


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=STAGE2_LR),
    loss={
        "carb_output": "mse",
        "mass_output": "mse",
    },
    loss_weights={
        "carb_output": 1.0,
        "mass_output": MASS_LOSS_WEIGHT,
    },
    metrics={
        "carb_output": [
            keras.metrics.MeanAbsoluteError(name="mae"),
            keras.metrics.MeanSquaredError(name="mse"),
        ],
        "mass_output": [
            keras.metrics.MeanAbsoluteError(name="mae"),
            keras.metrics.MeanSquaredError(name="mse"),
        ],
    },
    jit_compile=False
)

print("Stage 2 model compiled.")

Stage 2 model compiled.


In [ ]:
stage2_best_path = OUTPUT_DIR / "best_model_stage2.weights.h5"

stage2_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(stage2_best_path),
        monitor="val_carb_output_loss",
        save_best_only=True,
        save_weights_only=True,
        mode="min",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_carb_output_loss",
        patience=100,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_carb_output_loss",
        factor=0.5,
        patience=75,
        min_lr=1e-7,
        mode="min",
        verbose=1
    )
]

print("Stage 2 callbacks ready.")
print("Best Stage 2 weights will be saved to:")
print(stage2_best_path)

Stage 2 callbacks ready.
Best Stage 2 weights will be saved to:
/content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/best_model_stage2.weights.h5


In [ ]:
history_stage2 = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=STAGE2_EPOCHS,
    callbacks=stage2_callbacks,
    verbose=1
)

Epoch 1/200


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'pixel_values': 'pixel_values'}
Received: inputs=Tensor(shape=(None, 224, 224, 3))
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'images': 'images'}
Received: inputs=Tensor(shape=(None, 224, 224, 3))
  warnings.warn(msg)


119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 914ms/step - carb_output_loss: 526.2008 - carb_output_mae: 9.5831 - carb_output_mse: 526.2008 - loss: 1321.1149 - mass_output_loss: 15898.2820 - mass_output_mae: 42.2991 - mass_output_mse: 15898.2820
Epoch 1: val_carb_output_loss improved from None to 68.90096, saving model to /content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/best_model_stage2.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/best_model_stage2.weights.h5
119/119 ━━━━━━━━━━━━━━━━━━━━ 333s 1s/step - carb_output_loss: 222.9649 - carb_output_mae: 7.5160 - carb_output_mse: 222.9649 - loss: 550.7032 - mass_output_loss: 6554.7661 - mass_output_mae: 38.5433 - mass_output_mse: 6554.7661 - val_carb_output_loss: 68.9010 - val_carb_output_mae: 5.8295 - val_carb_output_mse: 70.7040 - val_loss: 206.9762 - val_mass_output_loss: 2647.2886 - val_mass_output_mae: 33.7665 - val_mass_output_mse:

KeyboardInterrupt: 

38 GB of GPU ram is used to train the model so it cant be made much larger.

Stopped the model due to the architecture not seeming that promising with its pseudo mass estimation.

Evaluating stage 2 on test set

In [ ]:
model.load_weights(str(stage2_best_path))
print("Loaded best Stage 2 weights.")

stage2_test_results = model.evaluate(test_dataset, verbose=1, return_dict=True)

print("\nStage 2 test results:")
print(stage2_test_results)

Loaded best Stage 2 weights.
26/26 ━━━━━━━━━━━━━━━━━━━━ 14s 517ms/step - carb_output_loss: 119.1757 - carb_output_mae: 6.2375 - carb_output_mse: 88.4835 - loss: 193.1603 - mass_output_loss: 2018.1008 - mass_output_mae: 29.5506 - mass_output_mse: 2093.5366

Stage 2 test results:
{'carb_output_loss': 119.17571258544922, 'carb_output_mae': 6.237484931945801, 'carb_output_mse': 88.48346710205078, 'loss': 193.16030883789062, 'mass_output_loss': 2018.100830078125, 'mass_output_mae': 29.550596237182617, 'mass_output_mse': 2093.53662109375}


Stage 2 carb prediction

In [ ]:
stage2_predictions = model.predict(test_dataset, verbose=1)["carb_output"].reshape(-1)

stage2_test_df = test_df.copy().reset_index(drop=True)
stage2_test_df["prediction_stage2"] = stage2_predictions
stage2_test_df["abs_error_stage2"] = np.abs(
    stage2_test_df["total_carb"] - stage2_test_df["prediction_stage2"]
)

display(stage2_test_df.head())

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'pixel_values': 'pixel_values'}
Received: inputs=Tensor(shape=(16, 224, 224, 3))
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'images': 'images'}
Received: inputs=Tensor(shape=(16, 224, 224, 3))
  warnings.warn(msg)


26/26 ━━━━━━━━━━━━━━━━━━━━ 60s 1s/step


,dish_id,total_carb,total_mass,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,side_c_start_path,side_c_mid_path,side_d_start_path,side_d_mid_path,prediction_stage2,abs_error_stage2
0,dish_1556575327,10.618,152.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,13.149956,2.531956
1,dish_1557861216,0.000,1.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,1.902302,1.902302
2,dish_1557862345,0.000,63.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,4.578474,4.578474
3,dish_1557862696,3.850,77.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,7.733685,3.883685
4,dish_1557862738,0.000,118.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,5.031718,5.031718


Top 10 best and worst for stage 2

In [ ]:
stage2_best_10 = stage2_test_df.sort_values("abs_error_stage2", ascending=True).head(10)
stage2_worst_10 = stage2_test_df.sort_values("abs_error_stage2", ascending=False).head(10)

stage2_best_10_csv = OUTPUT_DIR / "stage2_best_10.csv"
stage2_worst_10_csv = OUTPUT_DIR / "stage2_worst_10.csv"

stage2_best_10.to_csv(stage2_best_10_csv, index=False)
stage2_worst_10.to_csv(stage2_worst_10_csv, index=False)

display(stage2_best_10)
display(stage2_worst_10)

,dish_id,total_carb,total_mass,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,side_c_start_path,side_c_mid_path,side_d_start_path,side_d_mid_path,prediction_stage2,abs_error_stage2
16,dish_1557937715,4.680000,39.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,4.681336,0.001336
300,dish_1565117429,49.245003,147.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,49.280525,0.035522
60,dish_1558546434,0.396000,11.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,0.336497,0.059503
55,dish_1558461692,9.080000,40.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,9.018400,0.061600
146,dish_1561574848,4.815750,155.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,4.905969,0.090219
145,dish_1561574815,3.246844,89.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,3.133274,0.113570
288,dish_1565030350,2.292192,62.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,2.414122,0.121930
49,dish_1558376801,0.795000,15.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dat

,dish_id,total_carb,total_mass,split,source_cafe,rgb_path,side_a_start_path,side_a_mid_path,side_b_start_path,side_b_mid_path,side_c_start_path,side_c_mid_path,side_d_start_path,side_d_mid_path,prediction_stage2,abs_error_stage2
120,dish_1560801020,61.690983,250.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,17.108717,44.582266
119,dish_1560800988,61.249092,230.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,16.912708,44.336384
121,dish_1560801041,63.245892,292.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,21.461958,41.783934
91,dish_1560367952,45.956913,221.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,9.044154,36.912759
345,dish_1566328805,85.054001,488.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,52.632648,32.421353
400,dish_1568401302,69.864967,561.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,39.210735,30.654232
344,dish_1566328776,78.293999,436.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,48.421974,29.872025
92,dish_1560367980,47.243912,254.0,test,cafe1,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content/drive/MyDrive/Speciale/dataset/nutrit...,/content

Comparing stage 1 and stage 2

In [ ]:
comparison_df = pd.DataFrame({
    "stage": ["stage1", "stage2"],
    "test_total_loss": [stage1_test_results["loss"], stage2_test_results["loss"]],
    "test_carb_loss": [stage1_test_results["carb_output_loss"], stage2_test_results["carb_output_loss"]],
    "test_carb_mae":  [stage1_test_results["carb_output_mae"], stage2_test_results["carb_output_mae"]],
    "test_carb_mse":  [stage1_test_results["carb_output_mse"], stage2_test_results["carb_output_mse"]],
})

comparison_csv = OUTPUT_DIR / "stage1_stage2_comparison.csv"
comparison_df.to_csv(comparison_csv, index=False)

display(comparison_df)
print("Saved comparison to:")
print(comparison_csv)

,stage,test_total_loss,test_carb_loss,test_carb_mae,test_carb_mse
0,stage1,199.529160,95.652313,6.210975,80.951614
1,stage2,193.160309,119.175713,6.237485,88.483467


Saved comparison to:
/content/drive/MyDrive/Speciale/Models/Model_9_V1_DINOv2_RGB_plus_all_side_mass_aux/stage1_stage2_comparison.csv


Calculating PMAE on modified test set

In [ ]:
stage1_test_pmae = (stage1_test_results["carb_output_mae"] / test_df["total_carb"].mean()) * 100
stage2_test_pmae = (stage2_test_results["carb_output_mae"] / test_df["total_carb"].mean()) * 100

print("Stage 1 test PMAE:", stage1_test_pmae)
print("Stage 2 test PMAE:", stage2_test_pmae)

Stage 1 test PMAE: 30.225757652750136
Stage 2 test PMAE: 30.354767609151693


Plotting training and validation curves

In [ ]:
def plot_history(history, title_prefix=""):
    metrics = [
        "carb_output_loss",
        "carb_output_mae",
    ]

    for metric in metrics:
        plt.figure(figsize=(8, 5))
        plt.plot(history.history[metric], label=f"train_{metric}")
        plt.plot(history.history[f"val_{metric}"], label=f"val_{metric}")
        plt.title(f"{title_prefix} {metric}")
        plt.xlabel("Epoch")
        plt.ylabel(metric)
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
plot_history(history_stage1, title_prefix="Stage 1")

In [ ]:
plot_history(history_stage2, title_prefix="Stage 2")

In [ ]:
def plot_combined_history(hist1, hist2, metric="carb_output_loss", title=""):
    values = hist1.history[metric] + hist2.history[metric]
    val_values = hist1.history[f"val_{metric}"] + hist2.history[f"val_{metric}"]

    plt.figure(figsize=(8, 5))
    plt.plot(values, label=f"train_{metric}")
    plt.plot(val_values, label=f"val_{metric}")
    plt.title(title if title else f"Combined {metric}")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_combined_history(
    history_stage1,
    history_stage2,
    metric="carb_output_loss",
    title="Model 9 V1 - Combined Carb Loss"
)

In [ ]:
merged_test_df = test_df.copy().reset_index(drop=True)

merged_test_df["prediction_stage1"] = stage1_predictions
merged_test_df["abs_error_stage1"] = np.abs(
    merged_test_df["total_carb"] - merged_test_df["prediction_stage1"]
)

merged_test_df["prediction_stage2"] = stage2_predictions
merged_test_df["abs_error_stage2"] = np.abs(
    merged_test_df["total_carb"] - merged_test_df["prediction_stage2"]
)

merged_test_df["improvement_abs_error"] = (
    merged_test_df["abs_error_stage1"] - merged_test_df["abs_error_stage2"]
)

merged_pred_csv = OUTPUT_DIR / "test_predictions_stage1_vs_stage2.csv"
merged_test_df.to_csv(merged_pred_csv, index=False)

print("Saved merged test predictions to:")
print(merged_pred_csv)

display(merged_test_df.head())

Saving final model

In [ ]:
final_model_path = OUTPUT_DIR / "final_model_stage2.keras"
model.save(final_model_path)

print("Saved final model to:")
print(final_model_path)

Saving final weights

In [ ]:
final_weights_path = OUTPUT_DIR / "final_model_stage2.weights.h5"
model.save_weights(final_weights_path)

print("Saved final weights to:")
print(final_weights_path)

In [ ]:
run_summary = {
    "model_name": "model_9_v1_rgb_plus_all_side_mass_aux",
    "rgb_backbone": RGB_BACKBONE_PRESET,
    "side_backbone": SIDE_BACKBONE_PRESET,
    "img_height": IMG_HEIGHT,
    "img_width": IMG_WIDTH,
    "batch_size": BATCH_SIZE,
    "stage1_epochs": STAGE1_EPOCHS,
    "stage2_epochs": STAGE2_EPOCHS,
    "stage1_lr": STAGE1_LR,
    "stage2_lr": STAGE2_LR,
    "mass_loss_weight": MASS_LOSS_WEIGHT,
    "stage1_val_carb_loss": float(stage1_val_results["carb_output_loss"]),
    "stage1_val_carb_mae": float(stage1_val_results["carb_output_mae"]),
    "stage1_test_carb_loss": float(stage1_test_results["carb_output_loss"]),
    "stage1_test_carb_mae": float(stage1_test_results["carb_output_mae"]),
    "stage1_test_carb_pmae": float(stage1_test_pmae),
    "stage2_val_carb_loss": float(stage2_val_results["carb_output_loss"]),
    "stage2_val_carb_mae": float(stage2_val_results["carb_output_mae"]),
    "stage2_test_carb_loss": float(stage2_test_results["carb_output_loss"]),
    "stage2_test_carb_mae": float(stage2_test_results["carb_output_mae"]),
    "stage2_test_carb_pmae": float(stage2_test_pmae),
}

summary_path = OUTPUT_DIR / "run_summary.json"
with open(summary_path, "w") as f:
    json.dump(run_summary, f, indent=2)

print("Saved run summary to:")
print(summary_path)